# Laboratorio de Web Scraping
**Sitio objetivo:** [Books to Scrape](https://books.toscrape.com)

En este cuaderno aprenderás a:
1. Realizar solicitudes HTTP responsables con `requests`.
2. Analizar el HTML con `BeautifulSoup`.
3. Navegar por la paginación del sitio.
4. Construir un `DataFrame` con la información extraída.
5. Guardar los datos a CSV y hacer un análisis descriptivo rápido.

> **Recomendación docente**: ejecuta cada celda una vez entiendas el objetivo; revisa los comentarios en el código.

## 1. Prerrequisitos e importación de librerías
Instala las dependencias si aún no las tienes:

In [ ]:
!pip install -q requests beautifulsoup4 lxml pandas

In [ ]:
import requests, time, re
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

BASE_URL = 'https://books.toscrape.com/'
HEADERS = {'User-Agent': 'UTP-DataMiningLab-2025'}

print('Dependencias cargadas 🐍')

## 2. Función auxiliar para extraer datos de una sola página

In [ ]:
def scrape_page(url: str):
    """Descarga y devuelve lista de diccionarios con info de cada libro en la página"""
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'lxml')
    books = []

    # ── Manejo de errores ─────────────────────────────────────────────
    if resp.status_code == 404:
        print(f'⚠️  Página no encontrada: {url}')
        return [], None          # señal para construir la próxima URL
    resp.raise_for_status()      # para otros códigos de error (500, etc.)

    for article in soup.select('article.product_pod'):
        title = article.h3.a['title']
        price = float(article.select_one('.price_color').text[2:])  # quita símbolo £
        stock = article.select_one('.availability').text.strip()
        rating_class = article.select_one('p.star-rating')['class'][1]  # e.g. 'Three'
        rating = ['Zero','One','Two','Three','Four','Five'].index(rating_class)
        books.append({
            'title': title,
            'price_gbp': price,
            'rating': rating,
            'stock': stock,
            'scrape_ts': datetime.utcnow()
        })    
    return books

## 3. Navegar por todas las páginas

In [ ]:
BASE = "https://books.toscrape.com/catalogue/page-{}.html"

all_books = []                  # aquí acumularás los resultados

for page in range(1, 51):       # 1 … 50 inclusive
    url = BASE.format(page)
    print(f"Scrapeando página {page}: {url}")
    
    try:
        books = scrape_page(url)   # tu función debe devolver (lista_libros, siguiente_url)
        all_books.extend(books)
    except requests.HTTPError as e:   # por si aparece un 404 u otro error
        print(f"⚠️  Error {e.response.status_code} en {url}, se omite.")
        continue                      # salta a la siguiente iteración

print(f"Total de libros extraídos: {len(all_books)}")


## 4. Crear DataFrame y vista rápida

In [ ]:
df = pd.DataFrame(all_books)
df.head()

### Información general del DataFrame

In [ ]:
df.info()

## 5. Análisis descriptivo básico

In [ ]:
# Precio promedio y libro más caro
print('Precio medio (£):', df['price_gbp'].mean().round(2))
max_row = df.loc[df['price_gbp'].idxmax()]
print('Libro más caro:', max_row['title'], '£', max_row['price_gbp'])

# Distribución de ratings
rating_counts = df['rating'].value_counts().sort_index()
rating_counts

## 6. Guardar los datos

In [ ]:
df.to_csv('books_scraped.csv', index=False)
print('Archivo guardado como books_scraped.csv')

## 7. Reflexión ético/técnica
- Verifica los `robots.txt` de los sitios antes de extraer.
- Ajusta el `User-Agent` y respeta límites de peticiones.
- Para sitios con contenido cargado dinámicamente, considera Selenium o Playwright.

---
Fin del laboratorio 🚀